# 24. Normalization Test (Pipeline 05)

- Goal: check body-relative normalization and confirm only norm coordinates are produced here.
- Docs: `docs_eng/pipeline/05_normalization.md` / `docs/pipeline/05_normalization.md`
- Inputs: Preprocessed pose dataframe from prior-stage cells.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Normalization report, hip-center translation check, and torso-scale check.


In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go

from movement.io import load_pose_csv
from movement.config import LANDMARKS, CONNECTIONS
from movement.canonicalization import (
    CanonicalizationConfig,
    MovementPlaneAlignmentConfig,
    ProtocolHeightLateralWidthAlignmentConfig,
    apply_canonicalization,
)
from movement.floor_reference import FloorReferenceConfig
from movement.normalization import (
    normalize_pose_by_hip_torso,
    check_normalization_result,
)
from movement.visualization import (
    create_pose_animation,
    create_pose_comparison_animation,
)

from movement.pipeline import (
    ExerciseDefinitionConfig,
    NormalizationConfig,
    PipelineConfig,
    ValidationConfig,
    run_pipeline,
)


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"

df = load_pose_csv(csv_path)

def estimate_frame_duration_ms(dataframe, default_ms=33):
    if "timestamp" not in dataframe.columns:
        return default_ms
    dt = dataframe["timestamp"].astype(float).diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1, int(round(median_dt * 1000)))

frame_duration_ms = estimate_frame_duration_ms(df)
print(f"playback frame duration: {frame_duration_ms} ms (~{1000 / frame_duration_ms:.1f} fps)")

recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type='orthographic'),
)

def apply_recording_view_camera(fig):
    fig.update_layout(scene_camera=recording_view_camera)
    return fig

df.head()


In [ ]:
norm_df, norm_report = normalize_pose_by_hip_torso(
    df=df,
    landmarks=LANDMARKS,
)

print(json.dumps(norm_report, indent=2, ensure_ascii=False))


In [ ]:
check_report = check_normalization_result(norm_df)

print(json.dumps(check_report, indent=2, ensure_ascii=False))


In [ ]:
fig_compare = create_pose_comparison_animation(
    df=norm_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=("raw", "norm"),
    names=("Raw", "Normalized"),
    title="Raw vs Normalized Pose Coordinates",
    show_text=False,
    frame_duration=frame_duration_ms,
)

apply_recording_view_camera(fig_compare)
fig_compare.show()


## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
